# AGQO V3.2 research runs on Modal

This notebook is the entry point for every quantum simulation, benchmark rerun and live LLM experiment of
V3.2. It submits work to Modal and keeps a local copy of each run directory under `results/runs/`.
No circuit is simulated in this kernel: the functions live in `modal_research.py` (Modal CPU) and
`modal_app.py` (Qwen3.8-27B on a Modal H100).

Prerequisites: `modal token new` once (credentials stay in `~/.modal.toml`), the price snapshot at
`data/vn30/ohlcv_daily.csv`, and for LLM tasks the deployed `agqo-llm` app. Vietnamese guide:
`docs/vi/NGHIEN_CUU_MODAL.md`.

Set `TASKS` in the next cell (papermill parameter) to choose what runs.

In [1]:
# Parameters (papermill)
TASKS = ["smoke"]          # any of: smoke, reproduce, pytest, presets, account_benchmark, llm_mandates,
                           #         llm_circuits, knn_table, knn_study
PYTEST_ARGS = []
LLM_MODEL_APP = "agqo-llm"
RUN_TAG = ""

In [2]:
# Parameters
TASKS = "[\"pytest\",\"reproduce\"]"


In [3]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "modal_research.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import modal
import modal_research as mr
from agqo.research.orchestrate import keep, revision, show

REV = revision()
print("revision", REV)
kept = {}

revision {'sha': 'e3d9f2fce8c81681141e76ef4ef33232cd9d2673', 'short': 'e3d9f2f', 'dirty': True}


## Smoke test: image, data snapshot and Volume

In [4]:
if "smoke" in TASKS:
    with mr.app.run():
        res = mr.smoke.remote(REV)
    kept["smoke"] = keep(res); show(kept["smoke"])

## Product failure reproduction

Runs the decision pipeline on the edge cases reported against V3.1 and audits every result on the
post-trade NAV (cash after fees + holdings). Output: `product_failures.json`.

In [5]:
if "reproduce" in TASKS:
    with mr.app.run():
        res = mr.reproduce.remote(REV)
    kept["reproduce"] = keep(res); show(kept["reproduce"])
    pf = json.loads((kept["reproduce"] / "product_failures.json").read_text())
    for c in pf["cases"]:
        a = c.get("audit", {})
        print(f"{c['case'][:58]:58s} {c['outcome']:9s} status={c.get('status')} app_ok={c.get('app_all_ok')} audit_ok={a.get('all_ok')}")
    print("verifier accepts swapped numbers:", pf["verifier_swap"]["swapped"]["check"]["ok"])

20261120-071030-e3d9f2f-product-failures: status=ok seconds=18.5 errors=0
  exceptions: 0
  silent_cap_breaches: 0
  verifier_accepts_swap: False
  clarification_blocks_trades: False
reported: 200M, banking <= 25%                             decision  status=ok app_ok=True audit_ok=True
reported: hold at most one stock (20M)                     decision  status=ok app_ok=True audit_ok=True
hold at most one stock (100M)                              decision  status=ok app_ok=True audit_ok=True
hold at most two stocks (100M)                             decision  status=ok app_ok=True audit_ok=True
20M, banking <= 25%                                        decision  status=ok app_ok=True audit_ok=True
500M, banking <= 25%                                       decision  status=ok app_ok=True audit_ok=True
100M, each stock <= 10%                                    decision  status=ok app_ok=True audit_ok=True
20M, each stock <= 2% (below one lot of every stock)       decision  status=infeas

## Test suite (including the circuit-simulating tests) on Modal CPU

In [6]:
if "pytest" in TASKS:
    with mr.app.run():
        res = mr.pytest_suite.remote(REV, PYTEST_ARGS)
    kept["pytest"] = keep(res); show(kept["pytest"])
    print((kept["pytest"] / "pytest.txt").read_text()[-3000:])

20261120-071057-e3d9f2f-pytest: status=failed seconds=17.8 errors=0
  returncode: 1
  tail: 1 failed, 38 passed in 16.83s
.........................F.............                                  [100%]
=================================== FAILURES ===================================
________________ test_every_state_obeys_caps_on_post_trade_nav _________________

market = MarketData(close=ticker          ACB      BCM      BID  ...      VPB      VPL      VRE
date                           ...0   23300.0
2026-11-12   33200.0   12890.0   23390.0
2026-11-13   33350.0   12870.0   23890.0

[1209 rows x 3 columns])

    def test_every_state_obeys_caps_on_post_trade_nav(market):
        md = market
        d = md.rebalance_dates(EC.test_start, EC.test_end)[-1]
        uni, _ = md.universe(d, 250, 180)
        m = Mandate(sector_caps={"Banking": 0.25}, min_cash_pct=5.0)
        inst = build_rounding_instance(md, d, uni, Account(cash=200e6), EC, m)
        X = all_bitstrings(inst.n_qubits)
      

## Summary of this notebook execution

In [7]:
print(json.dumps({k: str(v) for k, v in kept.items()}, indent=1))

{
 "reproduce": "/Users/jant/Documents/ChatGPT/AI-Quantum Challenge 2026/.claude/worktrees/agqo-portfolio-allocation-d30055/results/runs/20261120-071030-e3d9f2f-product-failures",
 "pytest": "/Users/jant/Documents/ChatGPT/AI-Quantum Challenge 2026/.claude/worktrees/agqo-portfolio-allocation-d30055/results/runs/20261120-071057-e3d9f2f-pytest"
}
